# Generate Netlist Files from NPY Placement Objects

This notebook provides tools to convert .npy placement files into DEF (Design Exchange Format) and JSON netlist files.

## Features:
- Load placement data from .npy files
- Generate DEF files with cell placements
- Generate JSON format netlists
- Batch processing for multiple designs
- Validation and statistics

## 1. Import Required Libraries

In [ ]:
import numpy as np
import json
import os
from pathlib import Path
from datetime import datetime
import glob
from tqdm import tqdm

## 2. Configuration and Paths

In [ ]:
# Base directories
BASE_DIR = r"H:\Labs\Generative Ai\Ayush1\Ayush"
NPY_DIR_GCELL = os.path.join(BASE_DIR, "CircuitNet", "instance_placement_gcell-001", "instance_placement_gcell")
NPY_DIR_MICRON = os.path.join(BASE_DIR, "CircuitNet", "instance_placement_micron-002", "instance_placement_micron")
OUTPUT_DIR = os.path.join(BASE_DIR, "generated_netlists")

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"NPY Directory (GCell): {NPY_DIR_GCELL}")
print(f"NPY Directory (Micron): {NPY_DIR_MICRON}")
print(f"Output Directory: {OUTPUT_DIR}")
print(f"\nOutput directory created: {os.path.exists(OUTPUT_DIR)}")

## 3. Load and Inspect NPY Files

In [ ]:
def load_npy_placement(npy_path):
    """
    Load placement data from .npy file
    
    Returns:
        numpy array with placement coordinates
    """
    try:
        data = np.load(npy_path)
        return data
    except Exception as e:
        print(f"Error loading {npy_path}: {e}")
        return None

# List available NPY files
gcell_files = sorted(glob.glob(os.path.join(NPY_DIR_GCELL, "*.npy")))
micron_files = sorted(glob.glob(os.path.join(NPY_DIR_MICRON, "*.npy")))

print(f"Found {len(gcell_files)} GCell NPY files")
print(f"Found {len(micron_files)} Micron NPY files")
print(f"\nFirst 5 GCell files:")
for f in gcell_files[:5]:
    print(f"  {os.path.basename(f)}")

In [ ]:
# Inspect a sample NPY file
if gcell_files:
    sample_file = gcell_files[0]
    sample_data = load_npy_placement(sample_file)
    
    print(f"Sample file: {os.path.basename(sample_file)}")
    print(f"Data shape: {sample_data.shape}")
    print(f"Data type: {sample_data.dtype}")
    print(f"\nFirst 5 rows:")
    print(sample_data[:5])
    print(f"\nStatistics:")
    print(f"  Min values: {sample_data.min(axis=0)}")
    print(f"  Max values: {sample_data.max(axis=0)}")
    print(f"  Mean values: {sample_data.mean(axis=0)}")

## 4. Generate DEF (Design Exchange Format) Files

In [ ]:
def generate_def_file(placement_data, output_path, design_name, die_area=(0, 0, 1000000, 1000000)):
    """
    Generate a DEF file from placement data
    
    Args:
        placement_data: numpy array with shape (N, 2) or (N, 4) containing x, y coordinates
        output_path: path to save the DEF file
        design_name: name of the design
        die_area: tuple (x1, y1, x2, y2) defining the die area in database units
    """
    num_cells = placement_data.shape[0]
    
    # Normalize coordinates to die area if they're in [0, 1] range
    if placement_data.max() <= 1.0:
        x_coords = (placement_data[:, 0] * (die_area[2] - die_area[0]) + die_area[0]).astype(int)
        y_coords = (placement_data[:, 1] * (die_area[3] - die_area[1]) + die_area[1]).astype(int)
    else:
        x_coords = placement_data[:, 0].astype(int)
        y_coords = placement_data[:, 1].astype(int)
    
    with open(output_path, 'w') as f:
        # Header
        f.write(f"VERSION 5.8 ;\n")
        f.write(f"DIVIDERCHAR \"/\" ;\n")
        f.write(f"BUSBITCHARS \"[]\" ;\n")
        f.write(f"DESIGN {design_name} ;\n")
        f.write(f"UNITS DISTANCE MICRONS 2000 ;\n\n")
        
        # Die area
        f.write(f"DIEAREA ( {die_area[0]} {die_area[1]} ) ( {die_area[2]} {die_area[3]} ) ;\n\n")
        
        # Components section
        f.write(f"COMPONENTS {num_cells} ;\n")
        for i in range(num_cells):
            cell_name = f"cell_{i}"
            cell_type = "STDCELL"  # Default cell type
            x, y = x_coords[i], y_coords[i]
            orientation = "N"  # North orientation
            
            f.write(f"  - {cell_name} {cell_type}\n")
            f.write(f"    + PLACED ( {x} {y} ) {orientation} ;\n")
        
        f.write(f"END COMPONENTS\n\n")
        
        # End of design
        f.write(f"END DESIGN\n")
    
    print(f"DEF file generated: {output_path}")
    print(f"  Number of cells: {num_cells}")
    print(f"  Die area: {die_area}")

## 5. Generate JSON Netlist Files

In [ ]:
def generate_json_netlist(placement_data, output_path, design_name, metadata=None):
    """
    Generate a JSON netlist file from placement data
    
    Args:
        placement_data: numpy array with placement coordinates
        output_path: path to save the JSON file
        design_name: name of the design
        metadata: optional dictionary with additional metadata
    """
    num_cells = placement_data.shape[0]
    
    netlist = {
        "design_name": design_name,
        "timestamp": datetime.now().isoformat(),
        "num_cells": num_cells,
        "metadata": metadata or {},
        "cells": []
    }
    
    # Add cell information
    for i in range(num_cells):
        cell_info = {
            "id": i,
            "name": f"cell_{i}",
            "type": "STDCELL",
            "x": float(placement_data[i, 0]),
            "y": float(placement_data[i, 1]),
            "orientation": "N"
        }
        
        # Add additional dimensions if available
        if placement_data.shape[1] > 2:
            cell_info["width"] = float(placement_data[i, 2]) if placement_data.shape[1] > 2 else None
            cell_info["height"] = float(placement_data[i, 3]) if placement_data.shape[1] > 3 else None
        
        netlist["cells"].append(cell_info)
    
    # Write to JSON file
    with open(output_path, 'w') as f:
        json.dump(netlist, f, indent=2)
    
    print(f"JSON netlist generated: {output_path}")
    print(f"  Number of cells: {num_cells}")

## 6. Test on a Single NPY File

In [ ]:
# Select a test file
if gcell_files:
    test_file = gcell_files[0]
    test_data = load_npy_placement(test_file)
    
    # Extract design name from filename
    design_name = os.path.splitext(os.path.basename(test_file))[0]
    
    print(f"Testing with file: {os.path.basename(test_file)}")
    print(f"Design name: {design_name}")
    print(f"Data shape: {test_data.shape}\n")
    
    # Generate DEF file
    def_output = os.path.join(OUTPUT_DIR, f"{design_name}_placement.def")
    generate_def_file(test_data, def_output, design_name)
    
    print()
    
    # Generate JSON file
    json_output = os.path.join(OUTPUT_DIR, f"{design_name}_placement.json")
    metadata = {
        "source_file": os.path.basename(test_file),
        "data_shape": list(test_data.shape)
    }
    generate_json_netlist(test_data, json_output, design_name, metadata)
else:
    print("No NPY files found to test!")

## 7. Batch Process Multiple NPY Files

In [ ]:
def batch_process_npy_files(npy_files, output_dir, format='both', max_files=None):
    """
    Batch process multiple NPY files
    
    Args:
        npy_files: list of NPY file paths
        output_dir: directory to save output files
        format: 'def', 'json', or 'both'
        max_files: maximum number of files to process (None for all)
    """
    if max_files:
        npy_files = npy_files[:max_files]
    
    results = {
        'success': 0,
        'failed': 0,
        'total': len(npy_files)
    }
    
    print(f"Processing {len(npy_files)} NPY files...\n")
    
    for npy_file in tqdm(npy_files):
        try:
            # Load placement data
            placement_data = load_npy_placement(npy_file)
            if placement_data is None:
                results['failed'] += 1
                continue
            
            # Extract design name
            design_name = os.path.splitext(os.path.basename(npy_file))[0]
            
            # Generate DEF file
            if format in ['def', 'both']:
                def_output = os.path.join(output_dir, f"{design_name}_placement.def")
                generate_def_file(placement_data, def_output, design_name)
            
            # Generate JSON file
            if format in ['json', 'both']:
                json_output = os.path.join(output_dir, f"{design_name}_placement.json")
                metadata = {
                    "source_file": os.path.basename(npy_file),
                    "data_shape": list(placement_data.shape)
                }
                generate_json_netlist(placement_data, json_output, design_name, metadata)
            
            results['success'] += 1
            
        except Exception as e:
            print(f"\nError processing {os.path.basename(npy_file)}: {e}")
            results['failed'] += 1
    
    print(f"\n{'='*60}")
    print(f"Batch Processing Complete!")
    print(f"{'='*60}")
    print(f"Total files: {results['total']}")
    print(f"Successful: {results['success']}")
    print(f"Failed: {results['failed']}")
    print(f"Success rate: {results['success']/results['total']*100:.1f}%")
    
    return results

In [ ]:
# Process GCell files (limit to first 10 for testing)
print("Processing GCell NPY files...\n")
gcell_results = batch_process_npy_files(
    gcell_files, 
    OUTPUT_DIR, 
    format='both',
    max_files=10  # Remove or set to None to process all files
)

In [ ]:
# Process Micron files (limit to first 10 for testing)
print("\nProcessing Micron NPY files...\n")
micron_results = batch_process_npy_files(
    micron_files, 
    OUTPUT_DIR, 
    format='both',
    max_files=10  # Remove or set to None to process all files
)

## 8. Verify Generated Files

In [ ]:
# List generated files
generated_def_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.def")))
generated_json_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.json")))

print(f"Generated Files in {OUTPUT_DIR}:")
print(f"  DEF files: {len(generated_def_files)}")
print(f"  JSON files: {len(generated_json_files)}")
print(f"\nFirst 5 DEF files:")
for f in generated_def_files[:5]:
    print(f"  {os.path.basename(f)}")
print(f"\nFirst 5 JSON files:")
for f in generated_json_files[:5]:
    print(f"  {os.path.basename(f)}")

In [ ]:
# Inspect a sample generated JSON file
if generated_json_files:
    sample_json = generated_json_files[0]
    with open(sample_json, 'r') as f:
        sample_netlist = json.load(f)
    
    print(f"Sample JSON file: {os.path.basename(sample_json)}")
    print(f"\nDesign name: {sample_netlist['design_name']}")
    print(f"Number of cells: {sample_netlist['num_cells']}")
    print(f"Timestamp: {sample_netlist['timestamp']}")
    print(f"\nFirst 3 cells:")
    for cell in sample_netlist['cells'][:3]:
        print(f"  {cell}")

In [ ]:
# Inspect a sample generated DEF file
if generated_def_files:
    sample_def = generated_def_files[0]
    print(f"Sample DEF file: {os.path.basename(sample_def)}")
    print(f"\nFirst 30 lines:")
    print("=" * 60)
    with open(sample_def, 'r') as f:
        for i, line in enumerate(f):
            if i >= 30:
                break
            print(line.rstrip())

## 9. Advanced: Process with Custom Cell Names and Types

In [ ]:
def generate_def_with_metadata(placement_data, cell_names, cell_types, output_path, design_name):
    """
    Generate DEF file with custom cell names and types
    
    Args:
        placement_data: numpy array with placement coordinates
        cell_names: list of cell names
        cell_types: list of cell types
        output_path: path to save DEF file
        design_name: name of the design
    """
    num_cells = placement_data.shape[0]
    die_area = (0, 0, 1000000, 1000000)
    
    # Normalize coordinates
    if placement_data.max() <= 1.0:
        x_coords = (placement_data[:, 0] * (die_area[2] - die_area[0]) + die_area[0]).astype(int)
        y_coords = (placement_data[:, 1] * (die_area[3] - die_area[1]) + die_area[1]).astype(int)
    else:
        x_coords = placement_data[:, 0].astype(int)
        y_coords = placement_data[:, 1].astype(int)
    
    with open(output_path, 'w') as f:
        # Header
        f.write(f"VERSION 5.8 ;\n")
        f.write(f"DIVIDERCHAR \"/\" ;\n")
        f.write(f"BUSBITCHARS \"[]\" ;\n")
        f.write(f"DESIGN {design_name} ;\n")
        f.write(f"UNITS DISTANCE MICRONS 2000 ;\n\n")
        f.write(f"DIEAREA ( {die_area[0]} {die_area[1]} ) ( {die_area[2]} {die_area[3]} ) ;\n\n")
        
        # Components
        f.write(f"COMPONENTS {num_cells} ;\n")
        for i in range(num_cells):
            cell_name = cell_names[i] if cell_names and i < len(cell_names) else f"cell_{i}"
            cell_type = cell_types[i] if cell_types and i < len(cell_types) else "STDCELL"
            x, y = x_coords[i], y_coords[i]
            
            f.write(f"  - {cell_name} {cell_type}\n")
            f.write(f"    + PLACED ( {x} {y} ) N ;\n")
        
        f.write(f"END COMPONENTS\n\n")
        f.write(f"END DESIGN\n")
    
    print(f"Advanced DEF file generated: {output_path}")

## 10. Summary and Next Steps

In [ ]:
print("="*70)
print("NETLIST GENERATION SUMMARY")
print("="*70)
print(f"\nOutput Directory: {OUTPUT_DIR}")
print(f"\nGenerated Files:")
print(f"  DEF files: {len(glob.glob(os.path.join(OUTPUT_DIR, '*.def')))}")
print(f"  JSON files: {len(glob.glob(os.path.join(OUTPUT_DIR, '*.json')))}")
print(f"\nTo process all files, modify the max_files parameter in the batch processing cells.")
print(f"\nNext steps:")
print(f"  1. Validate DEF files with a DEF parser or viewer")
print(f"  2. Use JSON files for further analysis or visualization")
print(f"  3. Integrate with EDA tools for physical design flow")